# 02.6 Phase 2 Project: Digits Experiments

This notebook is the integrated project for Phase 2. The goal is to run a small comparative experiment, not just train one model. You compare an MLP baseline with a CNN, evaluate them under the same split, and inspect the errors of the best model.

The project structure matters because controlled comparison is the beginning of real experimentation. If data, metrics, or evaluation code change between models, the comparison becomes hard to trust.

## Learning Goals

After this notebook, you should be able to:

1. Build a small vision experiment project.
2. Compare MLP and CNN on an image task.
3. Use a shared training function for different models.
4. Organize comparison results into a result table.
5. Evaluate the best model and perform simple error analysis.
6. Think more like running a project rather than solving isolated exercises.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


## Prepare the Data

We continue using `digits` because it is small enough to support quick model-comparison experiments.


In [ ]:
digits = load_digits()
images = digits.images
labels = digits.target

X_train_full, X_test, y_train_full, y_test = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

print("train:", X_train.shape)
print("val:", X_val.shape)
print("test:", X_test.shape)

In [ ]:
train_mean = float(X_train.mean() / 16.0)
train_std = float(X_train.std() / 16.0)

transform = transforms.Compose([
    transforms.Lambda(lambda x: x.float() / 16.0),
    transforms.Normalize(mean=[train_mean], std=[train_std]),
])

In [ ]:
class DigitsProjectDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = torch.tensor(self.images[index], dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(int(self.labels[index]), dtype=torch.long)
        if self.transform is not None:
            image = self.transform(image)
        return image, label


train_ds = DigitsProjectDataset(X_train, y_train, transform=transform)
val_ds = DigitsProjectDataset(X_val, y_val, transform=transform)
test_ds = DigitsProjectDataset(X_test, y_test, transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

xb, yb = next(iter(train_loader))
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)

## Define Two Models

The goal here is to let both models share the same data pipeline and differ only in architecture.


In [ ]:
class MLPBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


print(MLPBaseline())
print()
print(CNNModel())

## Shared Training Function

To make the comparison fair, both models share the same training logic as much as possible.


In [ ]:
def batch_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_acc += batch_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


def train_model(model, train_loader, val_loader, epochs=5, lr=0.01):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
            }
        )

    return pd.DataFrame(history), model

## Train the MLP Baseline

An MLP makes a good baseline because it is simpler and faster.


In [ ]:
torch.manual_seed(0)
mlp_history, mlp_model = train_model(MLPBaseline(), train_loader, val_loader, epochs=5, lr=0.01)
print(mlp_history)

## Train the CNN

A CNN explicitly uses the spatial structure of images, which is its key advantage over an MLP.


In [ ]:
torch.manual_seed(0)
cnn_history, cnn_model = train_model(CNNModel(), train_loader, val_loader, epochs=5, lr=0.01)
print(cnn_history)

## Compare Validation Results

Here we put the final validation results of both models side by side.


In [ ]:
comparison = pd.DataFrame(
    {
        "model": ["MLPBaseline", "CNNModel"],
        "final_train_acc": [mlp_history.iloc[-1]["train_acc"], cnn_history.iloc[-1]["train_acc"]],
        "final_val_acc": [mlp_history.iloc[-1]["val_acc"], cnn_history.iloc[-1]["val_acc"]],
        "final_val_loss": [mlp_history.iloc[-1]["val_loss"], cnn_history.iloc[-1]["val_loss"]],
    }
)

print(comparison)

## Evaluate the Better Model on the Test Set

Here we simply choose the better model by final validation accuracy.


In [ ]:
best_model_name = comparison.sort_values("final_val_acc", ascending=False).iloc[0]["model"]
best_model = mlp_model if best_model_name == "MLPBaseline" else cnn_model

print("best model =", best_model_name)

In [ ]:
best_model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = best_model(xb)
        preds = logits.argmax(dim=1)
        all_preds.append(preds)
        all_targets.append(yb)

all_preds = torch.cat(all_preds)
all_targets = torch.cat(all_targets)

test_acc = accuracy_score(all_targets.numpy(), all_preds.numpy())
cm = confusion_matrix(all_targets.numpy(), all_preds.numpy())

print("test accuracy =", test_acc)
print("confusion matrix =\n", cm)

## Simple Error Analysis

Beyond a single overall accuracy number, we also want to see where the model makes mistakes.


In [ ]:
mis_idx = (all_preds != all_targets).nonzero(as_tuple=False).squeeze(1)
print("number of mistakes =", len(mis_idx))

In [ ]:
num_show = min(6, len(mis_idx))

if num_show > 0:
    plt.figure(figsize=(9, 4))
    for i in range(num_show):
        idx = mis_idx[i].item()
        image = X_test[idx]
        true_label = y_test[idx]
        pred_label = all_preds[idx].item()

        plt.subplot(2, 3, i + 1)
        plt.imshow(image, cmap="gray")
        plt.title(f"true={true_label}, pred={pred_label}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No mistakes to display.")

## Run Inference on New Samples

Here we run a simple inference demo using the first few images from the test set.


In [ ]:
sample_images = []
sample_labels = []

for i in range(6):
    image, label = test_ds[i]
    sample_images.append(image)
    sample_labels.append(label.item())

sample_batch = torch.stack(sample_images)

with torch.no_grad():
    logits = best_model(sample_batch)
    preds = logits.argmax(dim=1).tolist()

plt.figure(figsize=(9, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(sample_images[i].squeeze(0), cmap="gray")
    plt.title(f"true={sample_labels[i]}, pred={preds[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Exercise 1
#
# Experiment with MLP capacity.
#
# Change the MLP hidden size from 128 to 256 and compare the results again.
# Record whether validation accuracy improves, stays similar, or gets worse.
# Also consider whether the larger model is worth the extra parameters.

Exercise 1 Reference Note

This is an experiment exercise. A larger MLP hidden size increases capacity; compare whether validation accuracy improves enough to justify the extra parameters.

In [ ]:
# Exercise 2
#
# Add Dropout to the CNN and observe validation accuracy.
#
# When comparing, keep the rest of the CNN as similar as possible. If Dropout
# helps, the original model may have been overfitting. If it hurts, the task may
# already be simple enough that extra regularization is unnecessary.

Exercise 2 Reference Note

Dropout can help if the CNN overfits, but on a small and simple dataset it can also slightly hurt if the original model was already stable.

In [ ]:
# Exercise 3
#
# Answer in one or two full sentences:
# Why are CNNs often better suited than MLPs for image tasks?
#
# Your answer should mention local spatial structure and weight sharing.

Exercise 3 Reference Answer

CNNs are often better suited than MLPs for images because convolutions preserve local spatial structure and share weights across positions.

## Summary

The point of this project is controlled comparison. You built a shared data pipeline, trained more than one model, compared validation and test metrics, and inspected errors rather than stopping at a single accuracy number.

This is close to a real experiment workflow. A useful result is not just "model A is better"; it is an explanation of what changed, how it was measured, and what evidence supports the conclusion.